# Example: Operational Monitoring Dashboard

In this example, we build a REST polling dashboard for near-real-time portfolio monitoring, track escalation triggers on live Alpaca positions, and run what-if stress analysis on the current portfolio. We load the production engine results from Example 1 and monitor how the portfolio evolves.

> **By the end of this example, you will be able to:**
> * __Build a REST polling dashboard:__ Construct a polling loop that fetches account state, positions, and latest quotes from Alpaca every 30 seconds. Display real-time portfolio value, position weights, and unrealized P&L in a formatted dashboard.
> * __Monitor escalation triggers on live data:__ Run all three escalation checks (drawdown, sentiment crash, bandit churn) on each polling cycle. Display an escalation timeline showing trigger status over the monitoring period.
> * __Stress-test the live portfolio with what-if analysis:__ Apply hypothetical market shocks to the current portfolio composition and evaluate whether the escalation system would respond correctly. Quantify capital preserved under each stress scenario.

## Setup, Data and Prerequisites
We load session packages and connect to Alpaca. If the market is closed, we fall back to review mode using saved snapshots from a previous session.

In [ ]:
include("Include.jl");

### Constants


In [ ]:
# Dashboard and stress-test configuration
PRODUCTION_DATA_PATH = joinpath(_PATH_TO_DATA, "production-engine-results.jld2")
DASHBOARD_DATA_PATH = joinpath(_PATH_TO_DATA, "dashboard-results.jld2")
PRODUCTION_B0 = 10_000.0
ALLOCATION_EPSILON = 0.1
MAX_DRAWDOWN = 0.15
MAX_TURNOVER = 0.50
SENTIMENT_THRESHOLD = -0.5
SENTIMENT_OVERRIDE_LAMBDA = 2.0
MAX_BANDIT_CHURN = 2
N_POLLS = 5
POLL_INTERVAL_SECONDS = 30
STRESS_SCENARIOS = [("Market -10%", -0.10), ("Market -20%", -0.20), ("Market -30%", -0.30)]
SINGLE_NAME_STRESS = -0.50


In [ ]:
client, is_live, production_data_path, dashboard_data_path, my_tickers, sim_params, K, prod_history, event_history, prod_ctx = let
    # --- Step 1: Load Alpaca credentials ---
    creds_path = joinpath(_ROOT, "config", "credentials.toml");
    if !isfile(creds_path)
        error("Credentials not found. See config/credentials.toml.example.")
    end
    client = Alpaca.load_client(creds_path);

    clock = Alpaca.get_clock(client);
    is_live = clock.is_open;
    println("Market: $(is_live ? "OPEN" : "CLOSED")")

    # --- Step 2: Load production engine results from Example 1 ---
    production_data_path = PRODUCTION_DATA_PATH;
    dashboard_data_path = DASHBOARD_DATA_PATH;

    if isfile(production_data_path)
        saved = load(production_data_path);
        my_tickers = saved["tickers"];
        sim_params = saved["sim_params"];
        K = length(my_tickers);
        prod_history = haskey(saved, "history") ? saved["history"] : MyLiveProductionDayResult[];
        event_history = haskey(saved, "events") ? saved["events"] : MyEscalationEvent[];
        println("Loaded production history: $(length(prod_history)) days, $(length(event_history)) events");
    else
        error("No production engine results found. Run Example 1 first.")
    end

    # --- Step 3: Build production context ---
    prod_ctx = build(MyProductionContext, (
        tickers = my_tickers,
        sim_parameters = sim_params,
        B₀ = PRODUCTION_B0,
        epsilon = ALLOCATION_EPSILON,
        max_drawdown = MAX_DRAWDOWN,
        max_turnover = MAX_TURNOVER,
        sentiment_threshold = SENTIMENT_THRESHOLD,
        sentiment_override_lambda = SENTIMENT_OVERRIDE_LAMBDA,
        max_bandit_churn = MAX_BANDIT_CHURN
    ));
    return client, is_live, production_data_path, dashboard_data_path, my_tickers, sim_params, K, prod_history, event_history, prod_ctx
end


With the credentials verified and production history loaded, we build the polling dashboard.

___
## Task 1: REST Polling Dashboard
We fetch account state, positions, and latest quotes from Alpaca on a configurable polling schedule. Each snapshot captures the full portfolio state at a point in time.

> __Polling-based monitoring:__ We fetch account state, positions, and latest quotes from Alpaca every 30 seconds for a configurable number of cycles. Each snapshot is stored so we can reconstruct the intraday trajectory. In review mode, we load previously recorded snapshots.

In [ ]:
snapshots = let
    n_polls = N_POLLS;            # number of polling cycles
    interval_sec = POLL_INTERVAL_SECONDS;  # seconds between polls

    if is_live
        println("Starting polling dashboard: $(n_polls) polls, $(interval_sec)s interval...");
        snapshots = [];
        for i in 1:n_polls
            # --- Step 1: Fetch account, positions, and quotes ---
            acct = Alpaca.get_account(client);
            positions = Alpaca.list_positions(client);
            quotes = Dict{String,Any}();
            for sym in vcat(my_tickers, ["SPY"])
                quotes[sym] = Alpaca.get_latest_quote(client, sym);
            end
            spy_mid = (quotes["SPY"].ask_price + quotes["SPY"].bid_price) / 2.0;

            # --- Step 2: Build position dictionary ---
            pos_data = Dict{String,NamedTuple}();
            for ticker in my_tickers
                found = false;
                for p in positions
                    if p.symbol == ticker
                        pos_data[ticker] = (shares = p.qty, market_value = p.market_value,
                            unrealized_pl = p.unrealized_pl, current_price = p.current_price);
                        found = true;
                        break;
                    end
                end
                if !found
                    pos_data[ticker] = (shares = 0.0, market_value = 0.0,
                        unrealized_pl = 0.0, current_price = 0.0);
                end
            end

            # --- Step 3: Store snapshot ---
            push!(snapshots, (
                timestamp = string(now()),
                equity = acct.equity,
                cash = acct.cash,
                buying_power = acct.buying_power,
                positions = pos_data,
                spy_price = spy_mid,
                poll_num = i
            ));
            println("  Poll $(i)/$(n_polls): equity = \$$(round(acct.equity, digits=2))");

            if i < n_polls
                sleep(interval_sec);
            end
        end

        # --- Step 4: Save snapshots to disk ---
        save(dashboard_data_path, Dict(
            "snapshots" => snapshots,
            "tickers" => my_tickers,
            "timestamp" => string(now())
        ));
    elseif isfile(dashboard_data_path)
        saved = load(dashboard_data_path);
        snapshots = saved["snapshots"];
        println("REVIEW MODE: Loaded $(length(snapshots)) polling snapshots from $(saved["timestamp"])");
    else
        println("No dashboard data available. Run during market hours or after Example 1.");
        snapshots = [];
    end
    return snapshots
end


The code below displays the latest polling snapshot as account summary and holdings tables.

In [ ]:
let
    if !isempty(snapshots)
        snap = snapshots[end];

        # --- Step 1: Account summary table ---
        df_acct = DataFrame(
            "Metric" => ["Equity", "Cash", "Buying Power"],
            "Value (\$)" => [round(snap.equity, digits=2), round(snap.cash, digits=2),
                round(snap.buying_power, digits=2)]
        );
        println("Account Summary ($(snap.timestamp)):");
        pretty_table(df_acct; backend = :text,
            fit_table_in_display_horizontally = false,
            fit_table_in_display_vertically = false,
            table_format = TextTableFormat(borders = text_table_borders__compact))

        # --- Step 2: Holdings table ---
        total_value = sum(snap.positions[t].market_value for t in my_tickers) + snap.cash;
        df_pos = DataFrame(
            "Ticker" => my_tickers,
            "Shares" => [round(snap.positions[t].shares, digits=2) for t in my_tickers],
            "Price" => [round(snap.positions[t].current_price, digits=2) for t in my_tickers],
            "Value (\$)" => [round(snap.positions[t].market_value, digits=2) for t in my_tickers],
            "Weight (%)" => [round(snap.positions[t].market_value / total_value * 100, digits=1) for t in my_tickers],
            "Unrealized P&L" => [round(snap.positions[t].unrealized_pl, digits=2) for t in my_tickers]
        );
        println("\nHoldings:");
        pretty_table(df_pos; backend = :text,
            fit_table_in_display_horizontally = false,
            fit_table_in_display_vertically = false,
            table_format = TextTableFormat(borders = text_table_borders__compact))
    end
end

In [ ]:
let
    if length(snapshots) > 1
        # --- Step 1: Extract equity and compute drawdown ---
        equities = [s.equity for s in snapshots];
        W₀ = equities[1];
        scaled = equities ./ W₀;
        peak = accumulate(max, equities);
        drawdowns = (peak .- equities) ./ peak .* 100;

        # --- Step 2: Plot W/W₀ and drawdown ---
        p1 = plot(1:length(scaled), scaled, marker=:circle, markersize=4,
            label="W/W₀", linewidth=2, color=:steelblue,
            xlabel="Poll #", ylabel="W/W₀", title="Portfolio Value (scaled)",
            bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);

        p2 = plot(1:length(drawdowns), drawdowns, marker=:circle, markersize=4,
            label="Drawdown", linewidth=2, color=:coral,
            xlabel="Poll #", ylabel="Drawdown (%)", title="Session Drawdown",
            bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);

        plot(p1, p2, layout=(1, 2), size=(800, 350))
    else
        println("Need at least 2 polling snapshots to plot trends.")
    end
end

The equity and drawdown plots show the intraday trajectory captured by the polling loop. Next, we check escalation triggers on each snapshot.

___
## Task 2: Live Escalation Monitoring
We run all three escalation checks on each polling snapshot and display the results as a timeline. The timeline reveals whether the portfolio approached any trigger thresholds during the monitoring window.

> __Trigger timeline:__ We run all three escalation checks on each polling snapshot. The timeline shows whether the portfolio approached any trigger thresholds during the monitoring window, even if no trigger actually fired.

In [ ]:
trigger_timeline = let
    if !isempty(snapshots)
        # --- Step 1: Set up reference values ---
        prev_action = !isempty(prod_history) ? prod_history[end].bandit_action : ones(Int, K);
        peak_eq = maximum(s.equity for s in snapshots);

        # --- Step 2: Run escalation checks on each snapshot ---
        trigger_timeline = [];
        for snap in snapshots
            drawdown = (peak_eq - snap.equity) / peak_eq;
            sent = !isempty(prod_history) ? prod_history[end].sentiment : 0.0;

            dd_status = drawdown > prod_ctx.max_drawdown ? "CRITICAL" : "OK";
            sent_status = sent < prod_ctx.sentiment_threshold ? "WARNING" : "OK";
            churn_status = "OK";

            push!(trigger_timeline, (
                poll = snap.poll_num,
                timestamp = snap.timestamp,
                drawdown = round(drawdown * 100, digits=2),
                sentiment = round(sent, digits=3),
                dd_status = dd_status,
                sent_status = sent_status,
                churn_status = churn_status
            ));
        end

        # --- Step 3: Display trigger timeline ---
        df = DataFrame(
            "Poll" => [t.poll for t in trigger_timeline],
            "DD (%)" => [t.drawdown for t in trigger_timeline],
            "Sentiment" => [t.sentiment for t in trigger_timeline],
            "DD Trigger" => [t.dd_status for t in trigger_timeline],
            "Sent Trigger" => [t.sent_status for t in trigger_timeline],
            "Churn Trigger" => [t.churn_status for t in trigger_timeline]
        );
        println("Escalation Timeline:");
        pretty_table(df; backend = :text,
            fit_table_in_display_horizontally = false,
            fit_table_in_display_vertically = false,
            table_format = TextTableFormat(borders = text_table_borders__compact))
    end
    return trigger_timeline
end


The escalation timeline shows proximity to each trigger threshold across the monitoring window. Next, we stress-test the portfolio with hypothetical shocks.

___
## Task 3: What-If Stress Analysis
We apply hypothetical market shocks to the current portfolio and check which escalation triggers would fire. This validates the safety system on real positions with real dollar amounts.

> __Stress-testing live positions:__ We apply hypothetical market shocks to the current portfolio and check which escalation triggers would fire. Unlike the Monte Carlo backtest in Session 3 (which tested synthetic scenarios), this analysis operates on real positions with real dollar amounts. The question is: "if the market drops 20% right now, does the system protect capital?"

In [ ]:
stress_results = let
    if !isempty(snapshots)
        snap = snapshots[end];

        # --- Step 1: Get current prices and positions ---
        current_prices = [snap.positions[t].current_price for t in my_tickers];
        current_shares_stress = [snap.positions[t].shares for t in my_tickers];
        cash = snap.cash;
        peak = maximum(s.equity for s in snapshots);
        peak = max(peak, snap.equity);
        prev_action = !isempty(prod_history) ? prod_history[end].bandit_action : ones(Int, K);

        # --- Step 2: Define stress scenarios ---
        scenarios = MyStressScenario[];
        for (label, shock) in STRESS_SCENARIOS
            s = MyStressScenario();
            s.label = label;
            s.market_shock = shock;
            s.ticker_shocks = Dict{String,Float64}();
            push!(scenarios, s);
        end

        # --- Step 3: Add worst-stock scenario ---
        worst_k = argmin(current_prices);
        s = MyStressScenario();
        s.label = "$(my_tickers[worst_k]) -50%";
        s.market_shock = 0.0;
        s.ticker_shocks = Dict(my_tickers[worst_k] => SINGLE_NAME_STRESS);
        push!(scenarios, s);

        # --- Step 4: Run stress tests ---
        stress_results = MyStressResult[];
        for scenario in scenarios
            result = apply_stress_scenario(scenario, current_prices, current_shares_stress,
                cash, prod_ctx, peak, prev_action, my_tickers);
            push!(stress_results, result);
        end
    end
    return stress_results
end


The code below displays the stress test results and plots capital preserved under each scenario.

In [ ]:
let
    if @isdefined(stress_results) && !isempty(stress_results)
        snap = snapshots[end];
        current_value = snap.equity;

        # --- Step 1: Build stress results table ---
        df = DataFrame(
            "Scenario" => [r.scenario_label for r in stress_results],
            "Stressed Value (\$)" => [round(r.stressed_wealth, digits=0) for r in stress_results],
            "Drawdown (%)" => [round(r.drawdown * 100, digits=1) for r in stress_results],
            "Triggers" => [join([e.trigger_type for e in r.triggers_fired], ", ") for r in stress_results],
            "De-risk?" => [r.would_derisk ? "YES" : "NO" for r in stress_results],
            "Capital Preserved (\$)" => [round(r.capital_preserved, digits=0) for r in stress_results]
        );
        println("What-If Stress Analysis (current value: \$$(round(current_value, digits=0))):");
        pretty_table(df; backend = :text,
            fit_table_in_display_horizontally = false,
            fit_table_in_display_vertically = false,
            table_format = TextTableFormat(borders = text_table_borders__compact))
    end
end

In [ ]:
let
    if @isdefined(stress_results) && !isempty(stress_results)
        # --- Step 1: Extract labels and values ---
        labels = [r.scenario_label for r in stress_results];
        preserved = [r.capital_preserved for r in stress_results];
        snap = snapshots[end];

        # --- Step 2: Bar chart of capital preserved ---
        p = bar(labels, preserved, label="Capital Preserved",
            color=[:steelblue, :darkorange, :coral, :purple],
            xlabel="Scenario", ylabel="Value (\$)",
            title="Capital Preserved Under Stress",
            bg="gray95", background_color_outside="white", framestyle=:box,
            legend=false, rotation=15, size=(700, 400));
        hline!(p, [snap.equity], label="Current Value", linestyle=:dash, color=:black, linewidth=2);
        hline!(p, [snap.equity * 0.85], label="15% DD Limit", linestyle=:dot, color=:red, linewidth=1.5);
        p
    end
end

In [ ]:
let
    if is_live && @isdefined(stress_results)
        # --- Step 1: Save dashboard results ---
        saved = isfile(dashboard_data_path) ? load(dashboard_data_path) : Dict();
        saved["stress_results"] = stress_results;
        saved["trigger_timeline"] = trigger_timeline;
        saved["timestamp"] = string(now());
        save(dashboard_data_path, saved);
        println("Dashboard results saved.");
    end
end

___
## Summary
This example built an operational monitoring dashboard that polls Alpaca for portfolio state, runs escalation checks on each snapshot, and stress-tests the live portfolio with hypothetical shocks.

### Key Takeaways
* __Polling-based monitoring captures portfolio evolution:__ Fetching account state every 30 seconds produces a time series of equity and drawdown that reveals intraday portfolio dynamics. Each snapshot also provides position-level detail (shares, value, weight, unrealized P&L) for granular analysis.
* __Escalation timeline shows proximity to trigger thresholds:__ Even when no trigger fires, the timeline reveals how close the portfolio came to each limit during the monitoring window. This proximity information is more valuable than a binary pass/fail --- it indicates whether risk limits need tightening.
* __What-if stress analysis validates the safety system in real time:__ Applying hypothetical shocks to live positions and checking trigger responses is the production equivalent of the Session 3 backtest validation. The difference is that the positions, prices, and consequences are real --- a failed stress test means the escalation thresholds need immediate adjustment.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.